In [0]:
%sql
-- ============================================================
-- Agentic Restock Workflow — Mock Schema Bootstrap
-- ============================================================
-- Creates the ab_training.agentic_restock schema and all 5 tables
-- required by the multi-agent restocking pipeline.
-- ============================================================

CREATE SCHEMA IF NOT EXISTS ab_training.agentic_restock
COMMENT 'Mock schema for the Agentic Restock Workflow. Houses inventory stock levels, threshold config, consumption history, the quote/approval lifecycle table (open_request), and the fulfillment ledger (restock_requests). Designed for seamless swap to production data.';

In [0]:
%sql
-- ============================================================
-- Table 1: inventory_stock_level
-- Real-time stock snapshot per item/warehouse. Read by the
-- Lakeflow Job (coarse threshold check via JOIN to threshold_config)
-- and the Restock Agent (real-time validation before fulfillment).
-- NOTE: Threshold/target levels live in threshold_config_table,
--       NOT here. This table only tracks current state.
-- ============================================================

CREATE OR REPLACE TABLE ab_training.agentic_restock.inventory_stock_level (
  item_id STRING NOT NULL COMMENT 'Part SKU identifier, e.g. PRT-BRK-001',
  warehouse_id STRING NOT NULL COMMENT 'Warehouse code, e.g. WH-BLR-01',
  item_name STRING COMMENT 'Human-readable part/accessory name',
  category STRING COMMENT 'Part category: Engine, Brake, Electrical, Body, Suspension, Accessories, Fluid',
  part_number STRING COMMENT 'Manufacturer part number, e.g. BP-F200',
  current_stock_qty INT COMMENT 'Units currently in stock at this warehouse',
  unit_of_measure STRING COMMENT 'Unit of measure: UNITS, SETS, LITERS, PAIRS',
  last_updated_at TIMESTAMP COMMENT 'Last time this stock record was refreshed',
  CONSTRAINT pk_inventory_stock_level PRIMARY KEY (item_id, warehouse_id)
)
COMMENT 'Real-time stock snapshot per automotive part/warehouse. Contains only current state — threshold and target levels are defined in threshold_config_table and joined via (item_id, warehouse_id).';

-- Insert mock data (25 automotive parts/accessories across 5 warehouses)
INSERT INTO ab_training.agentic_restock.inventory_stock_level VALUES
  -- Engine parts
  ('PRT-ENG-001', 'WH-BLR-01', 'Oil Filter',              'Engine',       'OF-4501',   5, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-ENG-002', 'WH-DEL-01', 'Air Filter',              'Engine',       'AF-2200',   3, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-ENG-003', 'WH-BLR-01', 'Spark Plug Set (4pc)',    'Engine',       'SP-1100',  12, 'SETS',   '2026-08-14 06:00:00'),
  ('PRT-ENG-004', 'WH-MUM-01', 'Timing Belt',             'Engine',       'TB-3300',   8, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-ENG-005', 'WH-CHN-01', 'Cabin Air Filter',        'Engine',       'CAF-660',   7, 'UNITS',  '2026-08-14 06:00:00'),
  -- Brake parts
  ('PRT-BRK-001', 'WH-BLR-01', 'Ceramic Brake Pad Front', 'Brake',       'BP-F200',   4, 'SETS',   '2026-08-14 06:00:00'),
  ('PRT-BRK-002', 'WH-DEL-01', 'Ceramic Brake Pad Rear',  'Brake',       'BP-R201',  18, 'SETS',   '2026-08-14 06:00:00'),
  ('PRT-BRK-003', 'WH-CHN-01', 'Brake Rotor Front',       'Brake',       'BR-F100',  10, 'PAIRS',  '2026-08-14 06:00:00'),
  ('PRT-BRK-004', 'WH-MUM-01', 'Brake Caliper Assembly',  'Brake',       'BC-A300',   6, 'UNITS',  '2026-08-14 06:00:00'),
  -- Electrical parts
  ('PRT-ELC-001', 'WH-HYD-01', 'Car Battery 65Ah',        'Electrical',  'BAT-65A',   2, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-ELC-002', 'WH-BLR-01', 'Alternator',              'Electrical',  'ALT-1200',  7, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-ELC-003', 'WH-DEL-01', 'Starter Motor',           'Electrical',  'STM-800',  14, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-ELC-004', 'WH-HYD-01', 'Headlight Bulb H4',       'Electrical',  'HLB-H4',    9, 'PAIRS',  '2026-08-14 06:00:00'),
  ('PRT-ELC-005', 'WH-MUM-01', 'Ignition Coil',           'Electrical',  'IGC-440',  11, 'UNITS',  '2026-08-14 06:00:00'),
  -- Suspension parts
  ('PRT-SUS-001', 'WH-CHN-01', 'Shock Absorber Front',    'Suspension',  'SA-F500',   3, 'PAIRS',  '2026-08-14 06:00:00'),
  ('PRT-SUS-002', 'WH-BLR-01', 'Shock Absorber Rear',     'Suspension',  'SA-R501',  22, 'PAIRS',  '2026-08-14 06:00:00'),
  ('PRT-SUS-003', 'WH-HYD-01', 'Coil Spring Front',       'Suspension',  'CS-F100',  13, 'PAIRS',  '2026-08-14 06:00:00'),
  ('PRT-SUS-004', 'WH-MUM-01', 'Stabilizer Link',         'Suspension',  'SL-220',    2, 'PAIRS',  '2026-08-14 06:00:00'),
  -- Body parts
  ('PRT-BDY-001', 'WH-DEL-01', 'Side Mirror Assembly LH', 'Body',        'SM-L100',   6, 'UNITS',  '2026-08-14 06:00:00'),
  ('PRT-BDY-002', 'WH-BLR-01', 'Wiper Blade Set',         'Body',        'WB-2400',   8, 'SETS',   '2026-08-14 06:00:00'),
  ('PRT-BDY-003', 'WH-CHN-01', 'Headlight Assembly LH',   'Body',        'HL-L200',   6, 'UNITS',  '2026-08-14 06:00:00'),
  -- Fluid
  ('PRT-FLD-001', 'WH-CHN-01', 'Brake Fluid DOT4 1L',     'Fluid',       'BF-DOT4',   4, 'LITERS', '2026-08-14 06:00:00'),
  ('PRT-FLD-002', 'WH-HYD-01', 'Engine Oil 5W-30 4L',     'Fluid',       'EO-5W30',  16, 'LITERS', '2026-08-14 06:00:00'),
  -- Accessories
  ('PRT-ACC-001', 'WH-MUM-01', 'Floor Mat Set Universal',  'Accessories', 'FM-UNI-5',  5, 'SETS',   '2026-08-14 06:00:00'),
  ('PRT-ACC-002', 'WH-DEL-01', 'Seat Cover Set Premium',   'Accessories', 'SC-PRM-4', 19, 'SETS',   '2026-08-14 06:00:00');

In [0]:
%sql
-- ============================================================
-- Table 2: threshold_config_table
-- Reorder thresholds per item/warehouse. The Lakeflow Job joins
-- this with inventory_stock_level for the coarse trigger check.
-- ============================================================

CREATE OR REPLACE TABLE ab_training.agentic_restock.threshold_config_table (
  item_id STRING NOT NULL COMMENT 'Part SKU identifier (FK to inventory_stock_level)',
  warehouse_id STRING NOT NULL COMMENT 'Warehouse code (FK to inventory_stock_level)',
  reorder_point_qty INT COMMENT 'Stock level that triggers the Lakeflow Job coarse check (current_stock <= this value)',
  minimum_stock_qty INT COMMENT 'Absolute floor for CRITICAL urgency scoring by Genie Agent',
  target_stock_qty INT COMMENT 'Desired stock level post-restock; requested_qty = target - current',
  lead_time_days INT COMMENT 'Expected supplier fulfillment lead time in days',
  is_active BOOLEAN COMMENT 'Whether this threshold config is active (inactive configs are skipped by the trigger; default: true)',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  updated_at TIMESTAMP COMMENT 'Last modification timestamp',
  CONSTRAINT pk_threshold_config PRIMARY KEY (item_id, warehouse_id)
)
COMMENT 'Reorder threshold configuration per part/warehouse. Joined by the Lakeflow Job for coarse low-stock detection. The Genie Agent also reads this for urgency scoring.';

-- Insert matching config rows
-- Items with stock BELOW reorder_point (will trigger the Lakeflow Job):
--   ENG-001/BLR(5<8), ENG-002/DEL(3<10), BRK-001/BLR(4<8), BRK-004/MUM(6<10),
--   ELC-001/HYD(2<7), ELC-002/BLR(7<10), SUS-001/CHN(3<8), SUS-004/MUM(2<7),
--   BDY-001/DEL(6<9), BDY-003/CHN(6<8), FLD-001/CHN(4<8), ACC-001/MUM(5<9)
INSERT INTO ab_training.agentic_restock.threshold_config_table VALUES
  -- BELOW threshold (will trigger the Lakeflow Job)
  ('PRT-ENG-001', 'WH-BLR-01',  8, 3, 20, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ENG-002', 'WH-DEL-01', 10, 3, 18, 7, true,  current_timestamp(), current_timestamp()),
  ('PRT-BRK-001', 'WH-BLR-01',  8, 3, 15, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-BRK-004', 'WH-MUM-01', 10, 4, 18, 6, true,  current_timestamp(), current_timestamp()),
  ('PRT-ELC-001', 'WH-HYD-01',  7, 2, 15, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ELC-002', 'WH-BLR-01', 10, 4, 20, 6, true,  current_timestamp(), current_timestamp()),
  ('PRT-SUS-001', 'WH-CHN-01',  8, 3, 16, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-SUS-004', 'WH-MUM-01',  7, 2, 15, 4, true,  current_timestamp(), current_timestamp()),
  ('PRT-BDY-001', 'WH-DEL-01',  9, 3, 18, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-BDY-003', 'WH-CHN-01',  8, 3, 16, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-FLD-001', 'WH-CHN-01',  8, 3, 15, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ACC-001', 'WH-MUM-01',  9, 3, 18, 6, true,  current_timestamp(), current_timestamp()),
  -- ABOVE threshold (healthy stock, will NOT trigger)
  ('PRT-ENG-003', 'WH-BLR-01',  8, 4, 25, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ENG-004', 'WH-MUM-01',  7, 3, 20, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ENG-005', 'WH-CHN-01',  6, 3, 18, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-BRK-002', 'WH-DEL-01', 10, 5, 22, 7, true,  current_timestamp(), current_timestamp()),
  ('PRT-BRK-003', 'WH-CHN-01',  8, 4, 20, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ELC-003', 'WH-DEL-01',  8, 4, 22, 6, true,  current_timestamp(), current_timestamp()),
  ('PRT-ELC-004', 'WH-HYD-01',  7, 3, 18, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-ELC-005', 'WH-MUM-01',  8, 4, 20, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-SUS-002', 'WH-BLR-01', 10, 5, 25, 7, true,  current_timestamp(), current_timestamp()),
  ('PRT-SUS-003', 'WH-HYD-01',  8, 4, 20, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-BDY-002', 'WH-BLR-01',  7, 3, 20, 5, true,  current_timestamp(), current_timestamp()),
  ('PRT-FLD-002', 'WH-HYD-01',  8, 4, 20, 6, true,  current_timestamp(), current_timestamp()),
  ('PRT-ACC-002', 'WH-DEL-01', 10, 5, 24, 7, true,  current_timestamp(), current_timestamp());

num_affected_rows,num_inserted_rows
25,25


In [0]:
%sql
-- ============================================================
-- Table 3: consumption_history
-- Daily consumption records for the Genie Agent's trend
-- analysis, stockout forecasting, and urgency scoring.
-- ============================================================

CREATE OR REPLACE TABLE ab_training.agentic_restock.consumption_history (
  consumption_id BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Auto-generated surrogate key',
  item_id STRING NOT NULL COMMENT 'Part SKU identifier consumed',
  warehouse_id STRING NOT NULL COMMENT 'Warehouse where consumption occurred',
  consumption_date DATE NOT NULL COMMENT 'Date of consumption event',
  qty_consumed INT COMMENT 'Units consumed/used/transferred that day',
  consumption_type STRING COMMENT 'Type of consumption: PRODUCTION_USE, SERVICE_USE, RETAIL_SALE, TRANSFER, DAMAGE',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  CONSTRAINT pk_consumption_history PRIMARY KEY (consumption_id)
)
COMMENT 'Daily consumption records per part/warehouse. The Genie Agent reads the trailing 14 days to compute avg_daily_consumption, forecast stockout dates, and assign urgency levels.';

-- Generate 14 days of consumption data for 10 part/warehouse combos (140 rows)
-- Uses deterministic hash for reproducible pseudo-random quantities (1-5 units/day)
INSERT INTO ab_training.agentic_restock.consumption_history (item_id, warehouse_id, consumption_date, qty_consumed, consumption_type, created_at)
WITH items AS (
  SELECT item_id, warehouse_id FROM (
    VALUES
      ('PRT-ENG-001', 'WH-BLR-01'),
      ('PRT-ENG-002', 'WH-DEL-01'),
      ('PRT-BRK-001', 'WH-BLR-01'),
      ('PRT-ELC-001', 'WH-HYD-01'),
      ('PRT-ELC-002', 'WH-BLR-01'),
      ('PRT-SUS-001', 'WH-CHN-01'),
      ('PRT-SUS-004', 'WH-MUM-01'),
      ('PRT-BDY-001', 'WH-DEL-01'),
      ('PRT-ACC-001', 'WH-MUM-01'),
      ('PRT-FLD-001', 'WH-CHN-01')
  ) AS t(item_id, warehouse_id)
),
dates AS (
  SELECT explode(sequence(DATE'2026-08-01', DATE'2026-08-14', INTERVAL 1 DAY)) AS consumption_date
)
SELECT
  i.item_id,
  i.warehouse_id,
  d.consumption_date,
  -- Deterministic pseudo-random qty between 1 and 5
  CAST(abs(hash(concat(i.item_id, i.warehouse_id, cast(d.consumption_date AS STRING)))) % 5 + 1 AS INT) AS qty_consumed,
  -- ~60% PRODUCTION_USE, ~20% SERVICE_USE, ~10% RETAIL_SALE, ~5% TRANSFER, ~5% DAMAGE
  CASE
    WHEN abs(hash(concat(i.warehouse_id, cast(d.consumption_date AS STRING), i.item_id))) % 20 < 12 THEN 'PRODUCTION_USE'
    WHEN abs(hash(concat(i.warehouse_id, cast(d.consumption_date AS STRING), i.item_id))) % 20 < 16 THEN 'SERVICE_USE'
    WHEN abs(hash(concat(i.warehouse_id, cast(d.consumption_date AS STRING), i.item_id))) % 20 < 18 THEN 'RETAIL_SALE'
    WHEN abs(hash(concat(i.warehouse_id, cast(d.consumption_date AS STRING), i.item_id))) % 20 < 19 THEN 'TRANSFER'
    ELSE 'DAMAGE'
  END AS consumption_type,
  current_timestamp() AS created_at
FROM items i
CROSS JOIN dates d;

num_affected_rows,num_inserted_rows
140,140


In [0]:
%sql
-- ============================================================
-- Table 4: open_request
-- Quote/approval lifecycle table per architecture §6.1.
-- One row per quote generated by the Supervisor Agent after
-- the Genie Agent confirms restocking is needed.
-- ============================================================

CREATE OR REPLACE TABLE ab_training.agentic_restock.open_request (
  quote_id STRING NOT NULL COMMENT 'Unique quote identifier, e.g. QT-20260813-0001',
  request_status STRING COMMENT 'Lifecycle state: PENDING_APPROVAL, APPROVED, REJECTED, FULFILLING, NEEDS_REVIEW, COMPLETED',
  parts_requested STRING COMMENT 'JSON array of structs: [{item_id, item_name, warehouse_id, current_stock_qty, reorder_point_qty, requested_qty, unit_of_measure}]',
  urgency_level STRING COMMENT 'Urgency classification: CRITICAL, HIGH, MEDIUM, LOW',
  predicted_stockout_date DATE COMMENT 'Earliest predicted stockout across items in the quote',
  summary_report STRING COMMENT 'Genie Agent natural-language assumption/reasoning report',
  teams_message_id STRING COMMENT 'Reference ID of the Adaptive Card sent to Teams',
  teams_sent_at TIMESTAMP COMMENT 'When the Teams notification was dispatched',
  databricks_preview_url STRING COMMENT 'Deep link to the Databricks Review App for this quote',
  reviewed_by STRING COMMENT 'Email/ID of the PM who acted on the quote',
  decision STRING COMMENT 'Final decision: APPROVED or REJECTED',
  decision_at TIMESTAMP COMMENT 'Timestamp of the approval/rejection in the Databricks UI',
  decision_comments STRING COMMENT 'Optional approver comments explaining the decision',
  created_by STRING COMMENT 'Agent that created this quote record (default: supervisor_agent)',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  updated_at TIMESTAMP COMMENT 'Last modification timestamp',
  CONSTRAINT pk_open_request PRIMARY KEY (quote_id)
)
COMMENT 'Quote and approval lifecycle table. The Supervisor Agent writes quotes here after the Genie Agent confirms restocking need. Tracks the full lifecycle from PENDING_APPROVAL through COMPLETED/REJECTED.';

-- Insert 3 sample quotes in different lifecycle states
INSERT INTO ab_training.agentic_restock.open_request VALUES
  -- 1. COMPLETED: was approved and fulfilled
  (
    'QT-20260812-0001',
    'COMPLETED',
    '[{"item_id":"PRT-ENG-001","item_name":"Oil Filter","warehouse_id":"WH-BLR-01","current_stock_qty":5,"reorder_point_qty":8,"requested_qty":15,"unit_of_measure":"UNITS"},{"item_id":"PRT-BRK-001","item_name":"Ceramic Brake Pad Front","warehouse_id":"WH-BLR-01","current_stock_qty":4,"reorder_point_qty":8,"requested_qty":11,"unit_of_measure":"SETS"}]',
    'HIGH',
    DATE'2026-08-17',
    'Analysis: Both BLR warehouse parts show sustained high consumption (avg 3.2 units/day for Oil Filter, 2.8 sets/day for Brake Pads). Stockout predicted within 5 days at current production rate. Recommend immediate restock to target levels.',
    'msg-teams-abc123',
    '2026-08-12 09:30:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-20260812-0001',
    'pm@company.com',
    'APPROVED',
    '2026-08-12 10:15:00',
    'Approved. Critical for assembly line continuity at Bangalore plant.',
    'supervisor_agent',
    '2026-08-12 09:28:00',
    '2026-08-12 11:00:00'
  ),
  -- 2. PENDING_APPROVAL: awaiting PM review
  (
    'QT-20260814-0001',
    'PENDING_APPROVAL',
    '[{"item_id":"PRT-SUS-004","item_name":"Stabilizer Link","warehouse_id":"WH-MUM-01","current_stock_qty":2,"reorder_point_qty":7,"requested_qty":13,"unit_of_measure":"PAIRS"},{"item_id":"PRT-ACC-001","item_name":"Floor Mat Set Universal","warehouse_id":"WH-MUM-01","current_stock_qty":5,"reorder_point_qty":9,"requested_qty":13,"unit_of_measure":"SETS"},{"item_id":"PRT-BRK-004","item_name":"Brake Caliper Assembly","warehouse_id":"WH-MUM-01","current_stock_qty":6,"reorder_point_qty":10,"requested_qty":12,"unit_of_measure":"UNITS"}]',
    'CRITICAL',
    DATE'2026-08-16',
    'CRITICAL: Stabilizer Link at MUM-01 has only 2 pairs remaining with avg consumption of 2.5/day — stockout in <1 day. Floor Mat Sets and Brake Calipers also below threshold. Mumbai warehouse needs immediate attention for production continuity.',
    'msg-teams-def456',
    '2026-08-14 07:15:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-20260814-0001',
    NULL,
    NULL,
    NULL,
    NULL,
    'supervisor_agent',
    '2026-08-14 07:12:00',
    '2026-08-14 07:15:00'
  ),
  -- 3. REJECTED: PM decided not to restock
  (
    'QT-20260811-0001',
    'REJECTED',
    '[{"item_id":"PRT-FLD-001","item_name":"Brake Fluid DOT4 1L","warehouse_id":"WH-CHN-01","current_stock_qty":4,"reorder_point_qty":8,"requested_qty":11,"unit_of_measure":"LITERS"}]',
    'MEDIUM',
    DATE'2026-08-20',
    'Brake Fluid DOT4 at CHN-01 below reorder point. Avg consumption 1.8L/day suggests stockout in ~5 days. However, Chennai plant is switching to DOT5.1 specification next month per engineering directive.',
    'msg-teams-ghi789',
    '2026-08-11 08:00:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-20260811-0001',
    'pm@company.com',
    'REJECTED',
    '2026-08-11 09:30:00',
    'Rejected: DOT4 being phased out in favor of DOT5.1. Run down existing stock.',
    'supervisor_agent',
    '2026-08-11 07:55:00',
    '2026-08-11 09:30:00'
  );

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
-- ============================================================
-- Table 5: restock_requests
-- Fulfillment ledger. Written ONLY after PM approval AND
-- real-time stock validation by the Restock Agent succeeds.
-- ============================================================

CREATE OR REPLACE TABLE ab_training.agentic_restock.restock_requests (
  request_id STRING NOT NULL COMMENT 'Unique request identifier, e.g. RR-20260812-0001',
  quote_id STRING NOT NULL COMMENT 'FK to open_request.quote_id — links fulfillment to the approved quote',
  item_id STRING NOT NULL COMMENT 'Part SKU being restocked',
  item_name STRING COMMENT 'Human-readable part name',
  warehouse_id STRING NOT NULL COMMENT 'Destination warehouse for the restock',
  requested_qty INT COMMENT 'Quantity requested for restock (target - validated_stock)',
  validated_stock_qty INT COMMENT 'Actual stock at the moment the Restock Agent performed real-time validation',
  unit_of_measure STRING COMMENT 'Unit of measure: UNITS, SETS, LITERS, PAIRS',
  status STRING COMMENT 'Fulfillment status: CREATED, SUBMITTED, IN_TRANSIT, RECEIVED',
  supplier_reference STRING COMMENT 'External supplier/PO reference number',
  estimated_delivery_date DATE COMMENT 'Expected delivery date from supplier',
  created_at TIMESTAMP COMMENT 'Row creation timestamp (when Restock Agent wrote this)',
  updated_at TIMESTAMP COMMENT 'Last modification timestamp',
  CONSTRAINT pk_restock_requests PRIMARY KEY (request_id)
)
COMMENT 'Fulfillment ledger. Each row is a confirmed restock line item written by the Restock Agent after real-time stock validation. Linked to the approved quote via quote_id.';

-- Insert 2 sample restock requests tied to the COMPLETED quote (QT-20260812-0001)
INSERT INTO ab_training.agentic_restock.restock_requests VALUES
  (
    'RR-20260812-0001',
    'QT-20260812-0001',
    'PRT-ENG-001',
    'Oil Filter',
    'WH-BLR-01',
    15,
    5,
    'UNITS',
    'IN_TRANSIT',
    'PO-MANN-2026-08-4471',
    DATE'2026-08-19',
    '2026-08-12 11:00:00',
    '2026-08-13 14:00:00'
  ),
  (
    'RR-20260812-0002',
    'QT-20260812-0001',
    'PRT-BRK-001',
    'Ceramic Brake Pad Front',
    'WH-BLR-01',
    11,
    4,
    'SETS',
    'SUBMITTED',
    'PO-BREMBO-2026-08-1122',
    DATE'2026-08-20',
    '2026-08-12 11:00:00',
    '2026-08-12 11:00:00'
  );

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- ============================================================
-- Verification: confirm all tables created with expected row counts
-- ============================================================

SELECT 'inventory_stock_level' AS table_name, COUNT(*) AS row_count FROM ab_training.agentic_restock.inventory_stock_level
UNION ALL
SELECT 'threshold_config_table', COUNT(*) FROM ab_training.agentic_restock.threshold_config_table
UNION ALL
SELECT 'consumption_history', COUNT(*) FROM ab_training.agentic_restock.consumption_history
UNION ALL
SELECT 'open_request', COUNT(*) FROM ab_training.agentic_restock.open_request
UNION ALL
SELECT 'restock_requests', COUNT(*) FROM ab_training.agentic_restock.restock_requests
ORDER BY table_name;

table_name,row_count
consumption_history,140
inventory_stock_level,25
open_request,3
restock_requests,2
threshold_config_table,25


In [0]:
%sql
-- ============================================================
-- §4.1 Lakeflow Job — Coarse Low-Stock Check (Trigger Query)
-- This is the exact query the hourly Lakeflow Job will run.
-- If result set is non-empty, it invokes the Supervisor Agent
-- with these candidates as the payload.
-- ============================================================

SELECT
  isl.item_id,
  isl.item_name,
  isl.warehouse_id,
  isl.current_stock_qty,
  tct.reorder_point_qty,
  tct.target_stock_qty,
  (tct.target_stock_qty - isl.current_stock_qty) AS suggested_reorder_qty,
  tct.lead_time_days
FROM ab_training.agentic_restock.inventory_stock_level isl
JOIN ab_training.agentic_restock.threshold_config_table tct
  ON isl.item_id = tct.item_id AND isl.warehouse_id = tct.warehouse_id
WHERE tct.is_active = true
  AND isl.current_stock_qty <= tct.reorder_point_qty
ORDER BY (isl.current_stock_qty * 1.0 / tct.reorder_point_qty) ASC;

item_id,item_name,warehouse_id,current_stock_qty,reorder_point_qty,target_stock_qty,suggested_reorder_qty,lead_time_days
PRT-ELC-001,Car Battery 65Ah,WH-HYD-01,2,7,15,13,5
PRT-SUS-004,Stabilizer Link,WH-MUM-01,2,7,15,13,4
PRT-ENG-002,Air Filter,WH-DEL-01,3,10,18,15,7
PRT-SUS-001,Shock Absorber Front,WH-CHN-01,3,8,16,13,5
PRT-FLD-001,Brake Fluid DOT4 1L,WH-CHN-01,4,8,15,11,5
PRT-BRK-001,Ceramic Brake Pad Front,WH-BLR-01,4,8,15,11,5
PRT-ACC-001,Floor Mat Set Universal,WH-MUM-01,5,9,18,13,6
PRT-BRK-004,Brake Caliper Assembly,WH-MUM-01,6,10,18,12,6
PRT-ENG-001,Oil Filter,WH-BLR-01,5,8,20,15,5
PRT-BDY-001,Side Mirror Assembly LH,WH-DEL-01,6,9,18,12,5
